In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='model_training')
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'model_training')



In [4]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, ndcg_score
from rdkit.ML.Scoring.Scoring import CalcBEDROC
from rdkit.DataStructs import ExplicitBitVect

# ---------------------------------------------------------------------------
# Paths -- adjust to your environment.
# ---------------------------------------------------------------------------
KARAMAN_PATH = _kp('dataset/Karaman_Kinase_selectivity_filtered_withseq.csv')

# ---------------------------------------------------------------------------
MOL_SAE_LATENT_PATH = _kp('dataset/Karaman_10_50_filtered_SAE_latent_top95_features_from_merge_v4.csv')

# Kinase feature pipeline (matches the notebook's klifs_map workflow):
#   1) Read KLIFS_REF_PATH -> get the KLIFS bitstring for every kinase
#   2) Read KLIFS_LATENT_PATH -> SAE-compressed latent vector (one per kinase
#      row, same row order as KLIFS_REF_PATH, no header on row 0)
#   3) Build dict { KLIFS_bitstring -> latent_vector } and map each Karaman
#      row by its KLIFS bitstring
KLIFS_REF_PATH = _kp('feature_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv')
KLIFS_LATENT_PATH = _kp('feature_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv')
KLIFS_BITSTRING_COL = "KLIFS"   # column in KLIFS_REF_PATH holding the bitstring
KARAMAN_KLIFS_COL = "KLIFS"     # column in the Karaman file with the same bitstring

# One model path per CV fold. Per-fold metrics are aggregated as mean +/- std
# across folds (no probability averaging -- see PRINT_PER_FOLD_DETAIL below).
# Two sets of paths, selected by MOLECULE_FEATURE_MODE above.
MODEL_PATHS = [_kp(f'model/KinSPEC/KinSPEC_MCA_training_band_10_50_fold{i}_model_v4.json') for i in range(1, 6)]

MODE_TAG = "kinspec_mac"

# Where to write the CSV outputs. Tagged by mode so SAE and Morgan runs do
# not overwrite each other when the same script is reused.
OUTPUT_DIR = _kp('model_training/Karaman_MAC_v4_result')

# If True, additionally print the full per-molecule table for every fold.
# Leave False for manuscript-ready output; flip to True when debugging or
# checking fold-to-fold variance.
PRINT_PER_FOLD_DETAIL = False

# Morgan fingerprint column in the Karaman file.
MORGAN_FP_COL = "Morgan"
MORGAN_NBITS = 4096

# Top-K cutoffs to evaluate.
TOPK_LIST = [5, 10]

# BEDROC alpha. alpha=20 emphasises the top ~8% of the ranked list.
BEDROC_ALPHA = 20.0

# Bioactivity fill value used to mark non-measured (>10000 nM) entries.
# Used to split "full" vs "measured-only" Spearman.
FILL_VALUE = 10000.0

# ---------------------------------------------------------------------------
# Molecule grouping by number of active kinases (per molecule).
# Molecules with n_active outside [EXCLUDE_MIN_ACTIVE, EXCLUDE_MAX_ACTIVE) are
# reported individually but excluded from group-level aggregation:
#   - n_active == 0          : no signal (trivially excluded)
#   - n_active >= 94 (>=50%) : too promiscuous, binary classification meaningless
# The remaining molecules are split into three bins by absolute n_active:
GROUP_BINS = [
    ("selective",   1,  15),    # n_active in [1, 15)
    ("moderate",    15, 30),    # n_active in [15, 40)
    ("promiscuous", 30, 94),    # n_active in [40, 94)
]
EXCLUDE_LABEL_NO_ACTIVE = "EXCLUDED_no_active"
EXCLUDE_LABEL_TOO_PROMISCUOUS = "EXCLUDED_too_promiscuous"


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def bitstring_to_array(bitstr: str) -> np.ndarray:
    """Convert a '0101...' bitstring to a 1-D uint8 numpy array."""
    return np.frombuffer(bitstr.encode("ascii"), dtype=np.uint8) - ord("0")


def precision_at_k(y_true_sorted: np.ndarray, k: int) -> float:
    k = min(k, len(y_true_sorted))
    return float(y_true_sorted[:k].sum() / k)


def recall_at_k(y_true_sorted: np.ndarray, k: int, n_active: int) -> float:
    if n_active == 0:
        return float("nan")
    k = min(k, len(y_true_sorted))
    return float(y_true_sorted[:k].sum() / n_active)


def ndcg_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    # sklearn expects 2-D arrays for ndcg_score.
    return float(ndcg_score(y_true[None, :], y_score[None, :], k=k))


def bedroc(y_true: np.ndarray, y_score: np.ndarray, alpha: float) -> float:
    """BEDROC via RDKit. Expects descending sort by score."""
    order = np.argsort(-y_score, kind="stable")
    sorted_labels = y_true[order]
    # RDKit expects an iterable of [score-or-rank, label] pairs; only the
    # label is used after we sort externally.
    scored = [[1.0, int(lbl)] for lbl in sorted_labels]
    return float(CalcBEDROC(scored, col=1, alpha=alpha))


def iqr(x: np.ndarray) -> float:
    return float(np.percentile(x, 75) - np.percentile(x, 25))


def assign_group(n_active: int) -> str:
    """Map a molecule's n_active count to its group label."""
    if n_active == 0:
        return EXCLUDE_LABEL_NO_ACTIVE
    for label, lo, hi in GROUP_BINS:
        if lo <= n_active < hi:
            return label
    # Above the highest bin -> too promiscuous
    return EXCLUDE_LABEL_TOO_PROMISCUOUS


def evaluate_one_fold(df: pd.DataFrame, X: np.ndarray, model_path: str,
                      fold_label: str) -> pd.DataFrame:
    """
    Run one model on the prebuilt feature matrix X and return a per-molecule
    metrics dataframe with one row per molecule.
    """
    booster = xgb.Booster()
    booster.load_model(model_path)
    # Mirror the notebook: wrap X as a DataFrame with integer column names
    # so feature naming matches how the models were trained.
    X_df = pd.DataFrame(X)
    X_df.columns = range(X_df.shape[1])
    pred = booster.predict(xgb.DMatrix(X_df))
    df = df.copy()
    df["pred"] = pred

    rows = []
    for mol, sub in df.groupby("molecule_name", sort=False):
        y_true = sub["label"].to_numpy(dtype=int)
        y_score = sub["pred"].to_numpy(dtype=float)
        bioact = sub["bioactivity"].to_numpy(dtype=float)
        n_total = len(sub)
        n_active = int(y_true.sum())

        result = {"molecule_name": mol,
                  "n_kinases": n_total,
                  "n_active": n_active}

        if n_active == 0:
            print(f"[warn] [{fold_label}] {mol}: no actives, skipping")
            # Fill all metric fields with NaN so the dataframe column layout
            # is identical across the no-active and has-active branches.
            # Without this, n_active==0 rows would be missing metric columns
            # and downstream concat / grouping would emit ragged NaNs.
            result["AUROC"] = float("nan")
            result[f"BEDROC_a{int(BEDROC_ALPHA)}"] = float("nan")
            result["Spearman_full"] = float("nan")
            result["Spearman_measured"] = float("nan")
            result["n_measured"] = int((bioact < FILL_VALUE).sum())
            for k in TOPK_LIST:
                result[f"P@{k}"] = float("nan")
                result[f"R@{k}"] = float("nan")
                result[f"NDCG@{k}"] = float("nan")
            result["pred_std"] = float(np.std(y_score))
            result["pred_range"] = float(y_score.max() - y_score.min())
            result["pred_iqr"] = iqr(y_score)
            result["pred_mean"] = float(np.mean(y_score))
            rows.append(result)
            continue

        result["AUROC"] = float(roc_auc_score(y_true, y_score))
        result[f"BEDROC_a{int(BEDROC_ALPHA)}"] = bedroc(
            y_true, y_score, BEDROC_ALPHA
        )

        # Negate score so positive Spearman = correct ranking (high pred goes
        # with low bioactivity = high potency).
        rho_full, _ = spearmanr(bioact, -y_score)
        result["Spearman_full"] = float(rho_full) if not np.isnan(rho_full) else float("nan")

        measured_mask = bioact < FILL_VALUE
        if measured_mask.sum() >= 3:
            rho_meas, _ = spearmanr(bioact[measured_mask],
                                    -y_score[measured_mask])
            result["Spearman_measured"] = (
                float(rho_meas) if not np.isnan(rho_meas) else float("nan")
            )
        else:
            result["Spearman_measured"] = float("nan")
        result["n_measured"] = int(measured_mask.sum())

        order = np.argsort(-y_score, kind="stable")
        y_true_sorted = y_true[order]
        for k in TOPK_LIST:
            result[f"P@{k}"] = precision_at_k(y_true_sorted, k)
            result[f"R@{k}"] = recall_at_k(y_true_sorted, k, n_active)
            result[f"NDCG@{k}"] = ndcg_at_k(y_true, y_score, k)

        result["pred_std"] = float(np.std(y_score))
        result["pred_range"] = float(y_score.max() - y_score.min())
        result["pred_iqr"] = iqr(y_score)
        result["pred_mean"] = float(np.mean(y_score))
        rows.append(result)

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ---- Load Karaman long table ----
    # Force Morgan_FP to string -- pandas otherwise tries to coerce the
    # 4096-char bitstring to a numeric type and overflows.
    df = pd.read_csv(KARAMAN_PATH, dtype={MORGAN_FP_COL: str})
    required = {"molecule_name", "SMILES", "protein_name", "UniProt_ID",
                "bioactivity", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Karaman file missing columns: {missing}")

    print(f"Loaded Karaman: {len(df)} rows, "
          f"{df['molecule_name'].nunique()} molecules, "
          f"{df['UniProt_ID'].nunique()} kinases "
          f"mode_tag = {MODE_TAG})")

    # ---- Load KLIFS reference + SAE latent, build bitstring -> latent map ----
    # Matches the notebook's klifs_map() workflow exactly:
    #   - KLIFS_REF_PATH has one row per kinase with a 'KLIFS' bitstring column
    #   - KLIFS_LATENT_PATH has the SAE-compressed latent vector for each
    #     kinase, IN THE SAME ROW ORDER, and the original file's first row is
    #     a header that the notebook skips with skiprows=1.
    klifs_ref = pd.read_csv(KLIFS_REF_PATH, dtype={KLIFS_BITSTRING_COL: str})
    if KLIFS_BITSTRING_COL not in klifs_ref.columns:
        raise ValueError(
            f"KLIFS reference missing '{KLIFS_BITSTRING_COL}' column"
        )
    klifs_latent = pd.read_csv(KLIFS_LATENT_PATH, skiprows=1, header=None).values
    if len(klifs_ref) != len(klifs_latent):
        raise ValueError(
            f"KLIFS reference and latent file have different row counts: "
            f"{len(klifs_ref)} vs {len(klifs_latent)}"
        )

    klifs_keys = klifs_ref[KLIFS_BITSTRING_COL].astype(str).str.strip()
    klifs_dict = dict(zip(klifs_keys, klifs_latent))
    n_kinase_feats = klifs_latent.shape[1]
    print(f"Loaded KLIFS ref: {len(klifs_ref)} kinases, "
          f"SAE latent dim = {n_kinase_feats}")

    # ---- Resolve a KLIFS bitstring for every Karaman row ----
    # Preferred: Karaman file already has the 'KLIFS' column (notebook style).
    # Fallback: look it up via UniProt_ID using the KLIFS reference.
    if KARAMAN_KLIFS_COL in df.columns:
        karaman_klifs = df[KARAMAN_KLIFS_COL].astype(str).str.strip()
        print(f"Using Karaman's own '{KARAMAN_KLIFS_COL}' column for KLIFS lookup")
    else:
        if "UniProt_ID" not in klifs_ref.columns:
            raise ValueError(
                f"Karaman file has no '{KARAMAN_KLIFS_COL}' column and KLIFS "
                f"reference has no 'UniProt_ID' column for fallback lookup."
            )
        uni_to_klifs = dict(zip(
            klifs_ref["UniProt_ID"].astype(str),
            klifs_ref[KLIFS_BITSTRING_COL].astype(str).str.strip(),
        ))
        karaman_klifs = df["UniProt_ID"].astype(str).map(uni_to_klifs)
        n_missing = karaman_klifs.isna().sum()
        if n_missing:
            missing_uni = df.loc[karaman_klifs.isna(), "UniProt_ID"].unique()
            raise ValueError(
                f"{n_missing} Karaman rows have no KLIFS match via UniProt_ID. "
                f"Example UniProts: {list(missing_uni[:3])}"
            )
        print(f"Karaman has no '{KARAMAN_KLIFS_COL}' column; mapped via UniProt_ID")

    # Map every (resolved) bitstring to its SAE latent.
    kin_block = np.array(
        [klifs_dict.get(k) for k in karaman_klifs],
        dtype=object,
    )
    miss_mask = np.array([v is None for v in kin_block])
    if miss_mask.any():
        bad = karaman_klifs[miss_mask].unique()
        raise ValueError(
            f"{miss_mask.sum()} Karaman rows have a KLIFS bitstring not "
            f"present in KLIFS_LATENT_PATH. Example: {list(bad[:2])}"
        )
    kin_block = np.stack(kin_block).astype(np.float32)

    # ---- Build feature matrix once (shared across folds) ----
    # skiprows=1, header=None, aligned by row order to the Karaman table.
    mol_block = pd.read_csv(
        MOL_SAE_LATENT_PATH, skiprows=1, header=None
    ).to_numpy(dtype=np.float32)
    if len(mol_block) != len(df):
        raise ValueError(
            f"Mol SAE latent has {len(mol_block)} rows but Karaman has "
            f"{len(df)} rows -- row-order alignment broken."
        )
    print(f"Molecule features: SAE latent, shape={mol_block.shape}")


    X = np.hstack([mol_block, kin_block])

    print(f"Feature matrix shape: {X.shape} "
          f"(mol={mol_block.shape[1]}, kin={kin_block.shape[1]})")

    # ---- Evaluate every fold ----
    per_fold_tables = []
    for i, mp in enumerate(MODEL_PATHS, start=1):
        label = f"fold{i}"
        print(f"\n>>> Evaluating {label} ({Path(mp).name})")
        per_mol = evaluate_one_fold(df, X, mp, label)
        per_mol["fold"] = label
        per_fold_tables.append(per_mol)

    all_folds = pd.concat(per_fold_tables, ignore_index=True)

    # ---- Assign group to every molecule by n_active ----
    all_folds["group"] = all_folds["n_active"].apply(assign_group)

    # ---- Save raw long table (one row per molecule-fold) ----
    long_csv = os.path.join(
        OUTPUT_DIR, f"{MODE_TAG}_karaman_per_molecule_all_folds.csv"
    )
    all_folds.to_csv(long_csv, index=False)
    print(f"\nSaved long per-molecule table: {long_csv}")

    # ---- Per-fold detail (opt-in) ----
    pd.set_option("display.max_rows", None)
    pd.set_option("display.width", 200)
    pd.set_option("display.float_format", "{:.4f}".format)

    if PRINT_PER_FOLD_DETAIL:
        for fold_label, sub in all_folds.groupby("fold", sort=False):
            print()
            print("=" * 100)
            print(f"PER-MOLECULE METRICS  --  {fold_label}")
            print("=" * 100)
            print(sub.drop(columns=["fold"]).to_string(index=False))

    # ---- Cross-fold per-molecule table (mean +/- std across folds) ----
    metric_cols = [c for c in all_folds.columns
                   if c not in ("molecule_name", "n_kinases", "n_active",
                                "n_measured", "fold", "group")]

    grouped = all_folds.groupby("molecule_name", sort=False)
    mean_df = grouped[metric_cols].mean()
    std_df = grouped[metric_cols].std()
    meta = grouped[["n_kinases", "n_active", "group"]].first()

    # Sort: kept molecules first (by group order then by n_active),
    # then excluded ones at the bottom.
    bin_labels = [b[0] for b in GROUP_BINS]
    group_order_list = bin_labels + [EXCLUDE_LABEL_NO_ACTIVE,
                                     EXCLUDE_LABEL_TOO_PROMISCUOUS]
    is_excluded = meta["group"].isin(
        [EXCLUDE_LABEL_NO_ACTIVE, EXCLUDE_LABEL_TOO_PROMISCUOUS]
    )
    order_key = meta["group"].map(
        {lbl: i for i, lbl in enumerate(group_order_list)}
    )
    sort_idx = (
        pd.DataFrame({"order": order_key, "n_active": meta["n_active"]})
        .sort_values(["order", "n_active"])
        .index
    )

    # Wide CSV: one row per molecule, every metric split into _mean / _std
    # (matches the format of the PSICHIC / TransformerCPI / MolTrans /
    # HitScreen per-molecule scripts).
    wide_df = pd.DataFrame(index=sort_idx)
    wide_df["group"] = meta.loc[sort_idx, "group"]
    wide_df["n_kinases"] = meta.loc[sort_idx, "n_kinases"]
    wide_df["n_active"] = meta.loc[sort_idx, "n_active"]
    for c in metric_cols:
        wide_df[f"{c}_mean"] = mean_df.loc[sort_idx, c].values
        wide_df[f"{c}_std"] = std_df.loc[sort_idx, c].values
    wide_csv = os.path.join(
        OUTPUT_DIR, f"{MODE_TAG}_karaman_per_molecule_mean_std.csv"
    )
    wide_df.to_csv(wide_csv, index_label="molecule_name")
    print(f"Saved wide per-molecule mean/std table: {wide_csv}")

    # Pretty terminal table (mean±std string), unchanged from the original.
    formatted = pd.DataFrame(index=sort_idx)
    formatted["group"] = meta.loc[sort_idx, "group"]
    formatted["n_kinases"] = meta.loc[sort_idx, "n_kinases"]
    formatted["n_active"] = meta.loc[sort_idx, "n_active"]
    for c in metric_cols:
        formatted[c] = [
            f"{m:.4f}±{s:.4f}" if pd.notna(m) else "NaN"
            for m, s in zip(mean_df.loc[sort_idx, c],
                            std_df.loc[sort_idx, c])
        ]

    print()
    print("=" * 100)
    print(f"PER-MOLECULE METRICS  (mean ± std across {len(MODEL_PATHS)} folds)")
    print("=" * 100)
    print(formatted.to_string())

    # ---- Excluded molecules summary ----
    excluded = meta[is_excluded].copy()
    if len(excluded):
        print()
        print("=" * 100)
        print(f"EXCLUDED MOLECULES (excluded from group/overall aggregation)")
        print("=" * 100)
        for mol, row in excluded.iterrows():
            print(f"  - {mol:<25s} group={row['group']:<28s} "
                  f"n_active={row['n_active']}/{row['n_kinases']}")

    # ---- Group-level aggregation (only kept molecules) ----
    # Per Rex: average per-molecule metrics across molecules in a group,
    # also pooling across folds. So each (molecule, fold) is one observation;
    # then mean +/- std across those observations within a group.
    kept_folds = all_folds[~all_folds["group"].isin(
        [EXCLUDE_LABEL_NO_ACTIVE, EXCLUDE_LABEL_TOO_PROMISCUOUS]
    )].copy()

    # Build BOTH the numeric (CSV) and string (terminal) versions of the
    # group summary in one pass.
    group_summary_csv_rows = []
    group_summary_print_rows = []
    for label, _lo, _hi in GROUP_BINS:
        sub = kept_folds[kept_folds["group"] == label]
        n_mol = int(sub["molecule_name"].nunique())
        n_obs = int(len(sub))
        csv_row = {"group": label, "n_molecules": n_mol,
                   "n_obs(mol*fold)": n_obs}
        print_row = {"group": label, "n_molecules": n_mol,
                     "n_obs(mol×fold)": n_obs}
        for c in metric_cols:
            m = sub[c].mean()
            s = sub[c].std()
            csv_row[f"{c}_mean"] = float(m) if pd.notna(m) else float("nan")
            csv_row[f"{c}_std"]  = float(s) if pd.notna(s) else float("nan")
            print_row[c] = f"{m:.4f}±{s:.4f}" if pd.notna(m) else "NaN"
        group_summary_csv_rows.append(csv_row)
        group_summary_print_rows.append(print_row)

    group_summary_csv = pd.DataFrame(group_summary_csv_rows).set_index("group")
    group_summary_print = pd.DataFrame(group_summary_print_rows).set_index("group")

    group_csv_path = os.path.join(
        OUTPUT_DIR, f"{MODE_TAG}_karaman_group_summary.csv"
    )
    group_summary_csv.to_csv(group_csv_path)
    print(f"\nSaved group summary: {group_csv_path}")

    print()
    print("=" * 100)
    print(f"GROUP-LEVEL SUMMARY  "
          f"(mean ± std across molecule-fold observations within each group)")
    print("=" * 100)
    print(group_summary_print.to_string())

    # ---- Overall aggregate (kept molecules only) ----
    summary = pd.DataFrame({
        "mean":   kept_folds[metric_cols].mean(),
        "median": kept_folds[metric_cols].median(),
        "std":    kept_folds[metric_cols].std(),
        "q25":    kept_folds[metric_cols].quantile(0.25),
        "q75":    kept_folds[metric_cols].quantile(0.75),
    })
    summary["iqr"] = summary["q75"] - summary["q25"]

    overall_csv = os.path.join(
        OUTPUT_DIR, f"{MODE_TAG}_karaman_overall_summary.csv"
    )
    summary.to_csv(overall_csv, index_label="metric")
    print(f"\nSaved overall summary: {overall_csv}")

    print()
    print("=" * 100)
    print(f"OVERALL AGGREGATE  (kept molecules only: "
          f"{kept_folds['molecule_name'].nunique()} molecules × "
          f"{len(MODEL_PATHS)} folds = {len(kept_folds)} observations)")
    print("=" * 100)
    print(summary.to_string())

    # ---- Diagnostic interpretation hint ----
    print()
    print("=" * 100)
    print("DIAGNOSTIC HINT")
    print("=" * 100)
    print(
        "If the model truly uses kinase information, expect:\n"
        "  - High AUROC, BEDROC, P@K, NDCG@K\n"
        "  - Non-trivial pred_std / pred_range / pred_iqr per molecule\n"
        "If the model mostly reads the molecule (ignoring kinase), expect:\n"
        "  - pred_std small (~< 0.05) across most molecules\n"
        "  - AUROC / P@K near random-baseline for that molecule's active rate\n"
        "  - Active molecules: most kinases get a uniformly high score; the\n"
        "    model cannot tell active kinases from inactive ones."
    )


if __name__ == "__main__":
    main()

Loaded Karaman: 10298 rows, 38 molecules, 271 kinases mode_tag = kinspec_mac)
Loaded KLIFS ref: 482 kinases, SAE latent dim = 200
Karaman has no 'KLIFS' column; mapped via UniProt_ID
Molecule features: SAE latent, shape=(10298, 417)
Feature matrix shape: (10298, 617) (mol=417, kin=200)

>>> Evaluating fold1 (KinSPEC_MCA_training_band_10_50_fold1_model_v4.json)
[warn] [fold1] Lapatinib: no actives, skipping

>>> Evaluating fold2 (KinSPEC_MCA_training_band_10_50_fold2_model_v4.json)
[warn] [fold2] Lapatinib: no actives, skipping

>>> Evaluating fold3 (KinSPEC_MCA_training_band_10_50_fold3_model_v4.json)
[warn] [fold3] Lapatinib: no actives, skipping

>>> Evaluating fold4 (KinSPEC_MCA_training_band_10_50_fold4_model_v4.json)
[warn] [fold4] Lapatinib: no actives, skipping

>>> Evaluating fold5 (KinSPEC_MCA_training_band_10_50_fold5_model_v4.json)
[warn] [fold5] Lapatinib: no actives, skipping

Saved long per-molecule table: ./Karaman_MAC_v4_result/kinspec_mac_karaman_per_molecule_all_folds